In [9]:
import pandas as pd
import pandas.errors
import pandas.core.common
pandas.core.common.SettingWithCopyWarning = pandas.errors.SettingWithCopyWarning
from comorbidipy import comorbidity
import mysql.connector

cnx = mysql.connector.connect(user='root', password='Fbk!23SQLpw', port=3333,
                            host='127.0.0.1',
                            database='patient_events_selected_conditions')
cur = cnx.cursor()
cur.execute('''select * from inpatient_episode_history''')
df_history = pd.DataFrame(cur.fetchall())
df_history.columns = [item[0] for item in cur.description]
df_history.rename(columns={'episode_id' : 'id'}, inplace=True)
print(df_history.head())

# df_history = pd.read_csv("../data/inpatient_episode_history.csv")
def age_from_group(s: str) -> int:
    s = str(s)
    if s.endswith("+"):
        return int(s[:-1])
    if s == "0":
        return 0
    lo, hi = s.split("-")
    return (int(lo) + int(hi)) // 2 # "80-84" -> 82

df_history["age_group"] = df_history["age_group"].apply(age_from_group)
print(df_history["age_group"].head())

cci_df = comorbidity(
    df_history,
    id="id",                      # or "episode_id" if that's the actual name
    code="diagnosis_code",
    score="charlson",
    icd="icd9",
    variant="quan",
    weighting="charlson", 
    assign0=False,
    age="age_group"
)
print(cci_df.head().columns)
# Identify the binary comorbidity columns (usually 17-30 specific codes)
# We exclude 'id', scores, weighted scores (wscore_), and the original age column
print(cci_df.head())
# count number of comorbidities per patient
comorbidity_cols = [col for col in cci_df.columns if col not in ["id", "age_group","comorbidity_score","age_adj_comorbidity_score", 'survival_10yr']]
cci_df["num_comorbidities"] = cci_df[comorbidity_cols].sum(axis=1)
print(cci_df["num_comorbidities"].head())
# # extract only relevant columns: id, comorbidity_score, num_comorbidities
cci_df = cci_df[["id", "age_adj_comorbidity_score", "num_comorbidities"]]

create_table_query = """
CREATE TABLE IF NOT EXISTS inpatient_episode_comorbidity (
    id INT PRIMARY KEY,
    age_adj_comorbidity_score FLOAT,
    num_comorbidities INT
)
"""
cur.execute(create_table_query)
cnx.commit()

insert_query = """
INSERT INTO inpatient_episode_comorbidity (id, age_adj_comorbidity_score, num_comorbidities)
VALUES (%s, %s, %s)
"""
data_tuples = list(cci_df.itertuples(index=False, name=None))
cur.executemany(insert_query, data_tuples)
cnx.commit()

cur.close()
cnx.close()
# cci_df.to_csv("../data/inpatient_episode_comorbidity.csv", index=False)
# # take max comorbidity_score in the df
# max_cci = cci_df["age_adj_comorbidity_score"].max()
# print(f"Max Charlson Comorbidity Index: {max_cci}")

       id diagnosis_code age_group
0  188664            412     65-69
1  188664         427.32     65-69
2  188664         V45.81     65-69
3  188683          484.8     30-34
4  188683         493.00     30-34
0    67
1    67
2    67
3    32
4    32
Name: age_group, dtype: int64
Index(['id', 'age_group', 'aids', 'ami', 'canc', 'cevd', 'chf', 'copd',
       'dementia', 'hp', 'metacanc', 'mld', 'pud', 'pvd', 'rend', 'rheumd',
       'diab', 'diabwc', 'msld', 'comorbidity_score',
       'age_adj_comorbidity_score', 'survival_10yr'],
      dtype='object')
       id  age_group  aids  ami  canc  cevd  chf  copd  dementia   hp  ...  \
0  188664         67   0.0  1.0   0.0   0.0  0.0   0.0       0.0  0.0  ...   
1  188683         32   0.0  0.0   0.0   0.0  0.0   1.0       0.0  0.0  ...   
2  188696         77   0.0  0.0   0.0   0.0  0.0   0.0       0.0  0.0  ...   
3  188743         82   0.0  0.0   0.0   0.0  0.0   0.0       0.0  0.0  ...   
4  188744         82   0.0  0.0   0.0   0.0  1.0   0